In [ ]:
import pandas as pd
import requests

In [ ]:
url = "https://earthquake.usgs.gov/fdsnws/event/1/query"
params = {
    "format": "geojson",
    "eventtype": "earthquake",
    "minmagnitude": 2.0,
    "starttime": "2026-10-01T00:00:00",
    "endtime": "2026-10-31T23:59:59.999",
}
response = requests.get(url, params=params, timeout=60)
response.raise_for_status()
data = response.json()

In [ ]:
df = pd.json_normalize(data["features"])
N = len(df)
print(f"Target population size (N): {N}")
df.head()

## Convenience sample
Select the first 100 rows in the order returned by the population query.

In [ ]:
# The population query uses the default USGS order (newest first).
convenience_sample = df.iloc[:100].copy()
convenience_sample.head()

## Simple random sample
Select 100 rows from the full population without replacement, using a fixed seed.

In [ ]:
random_sample = df.sample(n=100, random_state=42)
random_sample.head()

## Mean magnitude and time span
The population mean is μ. Each sample mean is x̄. A positive difference indicates an overestimate of μ. Time span runs from the earliest to the latest earthquake in each group.

In [ ]:
mu = df["properties.mag"].mean()

def summarize(frame):
    times = pd.to_datetime(frame["properties.time"], unit="ms", utc=True)
    mean = frame["properties.mag"].mean()
    return {
        "Rows": len(frame),
        "Mean magnitude": mean,
        "Difference from population mean": mean - mu,
        "Earliest (UTC)": times.min(),
        "Latest (UTC)": times.max(),
        "Time span (days)": (times.max() - times.min()).total_seconds() / 86400,
    }

comparison = pd.DataFrame.from_dict({
    "Population": summarize(df),
    "Convenience sample": summarize(convenience_sample),
    "Random sample": summarize(random_sample),
}, orient="index")
comparison.round(4)

## Why can the convenience sample distort the estimate?

The API returns the newest earthquakes first. Taking the first 100 gives the most recent events all the weight and leaves earlier events out. If an aftershock sequence produced many small earthquakes near the end of October, for example, it could pull the convenience-sample mean below μ. A period with larger earthquakes could push it above μ. The sampling method alone does not determine the direction of the error.

In the simple random sample, every earthquake in the monthly population has the same chance of selection. Its mean is unbiased across repeated draws, although this particular draw can still differ from μ.

## Stratified sample

The region is the final part of the USGS place label. Four broad groups are used, with unmatched regions included in Other. The largest-remainder method allocates exactly 200 rows in proportion to population size.

In [ ]:
import numpy as np

df["region"] = df["properties.place"].fillna("Unknown").str.rsplit(",", n=1).str[-1].str.strip()

north_america = r"alaska|california|nevada|hawaii|washington|oregon|idaho|utah|montana|wyoming|texas|new mexico|arizona|oklahoma|puerto rico|virgin islands|canada|mexico|guatemala|honduras|el salvador|nicaragua|costa rica|panama|caribbean|cuba|dominican|haiti|jamaica|bering|aleutian|^ak$|^ca$|^nv$|^hi$|^pr$|^us$"
asia = r"japan|indonesia|philippines|china|taiwan|russia|kamchatka|kuril|india|pakistan|afghanistan|iran|turkey|türkiye|myanmar|thailand|nepal|tibet|mongolia|uzbekistan|tajikistan|kyrgyz|kazakhstan|israel|iraq|indian ocean|mariana"
south_america = r"chile|argentina|peru|ecuador|colombia|bolivia|venezuela|brazil|south sandwich"
df["stratum"] = np.select(
    [df["region"].str.contains(pattern, case=False, regex=True) for pattern in (north_america, asia, south_america)],
    ["North America", "Asia", "South America"],
    default="Other regions",
)

n = 200
counts = df.groupby("stratum").size()
quotas = counts * n / len(df)
allocation = np.floor(quotas).astype(int)
remainder = n - allocation.sum()
allocation.loc[(quotas - allocation).sort_values(ascending=False).index[:remainder]] += 1
rng = np.random.default_rng(42)
stratified_sample = pd.concat([
    group.sample(n=int(allocation.loc[name]), random_state=rng)
    for name, group in df.groupby("stratum")
])
allocation_table = pd.DataFrame({"Population": counts, "Population share": counts / len(df), "Sample rows": allocation})
print(allocation_table)
print("Total sampled:", len(stratified_sample))
stratified_sample[["region", "stratum", "properties.mag"]].head()

## Variance of the mean magnitude

The following calculation estimates the sampling variance of each mean, including the finite-population correction. The stratified estimator weights each stratum mean by its population share. The earlier simple random sample contains 100 rows, so its variance is calculated using that sample size. The displayed result identifies which estimate is smaller for these samples. Magnitude variation within each sample is different from the variance of its estimated mean.

In [ ]:
stratum_stats = stratified_sample.groupby("stratum")["properties.mag"].agg(["mean", "var", "count"])
weights = counts / len(df)
stratified_mean = (weights * stratum_stats["mean"]).sum()
stratified_variance = (weights**2 * (1 - allocation / counts) * stratum_stats["var"] / allocation).sum()
srs_variance = (1 - len(random_sample) / len(df)) * random_sample["properties.mag"].var() / len(random_sample)
print(f"Stratified mean: {stratified_mean:.4f}; estimated variance: {stratified_variance:.6f}")
print(f"Simple random mean: {random_sample['properties.mag'].mean():.4f}; estimated variance: {srs_variance:.6f}")
print("The estimated variance of the stratified mean is", "lower" if stratified_variance < srs_variance else "higher" if stratified_variance > srs_variance else "equal to", "the variance of the simple random mean.")